# Data Wrangling - Part 2: Transforming, Summarizing, Joining, and Reshaping Data

This section introduces the core concept. Focus on the examples below and experiment by modifying the code to reinforce your understanding.

## Why these wrangling skills matter

This section introduces the core concept. Focus on the examples below and experiment by modifying the code to reinforce your understanding.

---
## Set up your session

We will continue using the North Temperate Lakes Long-Term Ecological Research dataset used in the previous notebook.

In [ ]:
# Packages
from pathlib import Path
import pandas as pd

In [ ]:
# Paths
root_fldr = Path.cwd().parent
raw_fldr = root_fldr / "data" / "raw"
processed_fldr = root_fldr / "data" / "processed"

In [ ]:
# Data import
NTL_phys_data = pd.read_csv(
    raw_fldr / "NTL-LTER_Lake_ChemistryPhysics_Raw.csv",
    dtype={
        "lakeid": "category",
        "lakename": "category",
    },
    parse_dates=["sampledate"],
    date_format="%m/%d/%y"
)

NTL_phys_data.head()


In [ ]:
# Inspect columns, data types, and non-null counts
NTL_phys_data.info()

## 1. Selecting columns

This section introduces the core concept. Focus on the examples below and experiment by modifying the code to reinforce your understanding.

In [ ]:
# List all column names
NTL_phys_data.columns

In [ ]:
# Select one column
NTL_phys_data["lakename"].head()

In [ ]:
# Select multiple columns
core_cols = ["lakename", "sampledate", "depth", "temperature_C", "dissolvedOxygen"]

lake_core = NTL_phys_data[core_cols]

lake_core.head()

### Selecting columns with `.loc[]`

`.loc[]` can select rows and columns at the same time.

The general pattern is:

```python
df.loc[row_filter, column_selection]
```

The colon `:` means "all rows" or "all columns," depending on where it appears.

In [ ]:
# Select all rows but only a few columns
lake_core_alt = NTL_phys_data.loc[:, core_cols]

lake_core_alt.head()

In [ ]:
# Select surface observations and a limited set of columns
surface_core = NTL_phys_data.loc[
    NTL_phys_data["depth"] == 0,
    core_cols
]

surface_core.head()

### Exercise 1

Create a dataframe named `oxygen_data` that contains only these columns (in this order):

- `sampledate`
- `lakename`
- `depth`
- `dissolvedOxygen`

Then preview the first five rows.

In [ ]:
# Exercise 1

---
## 2. Renaming variables

Raw datasets often contain variable names that are abbreviated, inconsistent, or difficult to remember.

In pandas, we commonly rename columns with `.rename()`:

```python
df.rename(columns={"old_name": "new_name"})
```

By default, `.rename()` returns a modified copy. It does not permanently change the original dataframe unless you assign the result to a variable.

In [ ]:
# Rename selected columns for readability
lake_core_renamed = lake_core.rename(
    columns={
        "lakename": "lake_name",
        "sampledate": "sample_date",
        "dissolvedOxygen": "dissolved_oxygen",
    }
)

lake_core_renamed.head()

### A note on naming conventions

For analysis workflows, consistent column names are more important than perfect column names.

A common Python-friendly convention is **snake_case**:

```text
lake_name
sample_date
dissolved_oxygen
```

Avoid spaces and special characters in column names when possible. They make code harder to write and can create problems in downstream tools.

In [ ]:
# Compare original and renamed columns
print("Original columns:")
print(list(lake_core.columns))

print("\nRenamed columns:")
print(list(lake_core_renamed.columns))

### Exercise 2

Create a renamed version of `oxygen_data` with the following column names:

- `lake_name`
- `sample_date`
- `depth_m`
- `dissolved_oxygen_mg_L`

Save the result as `oxygen_data_renamed`.

In [ ]:
# Exercise 2

## 3. Creating new variables

This section introduces the core concept. Focus on the examples below and experiment by modifying the code to reinforce your understanding.

In [ ]:
# Create a copy before adding new variables
lake_features = lake_core_renamed.copy()

# Extract date components
lake_features["year"] = lake_features["sample_date"].dt.year
lake_features["month"] = lake_features["sample_date"].dt.month

lake_features.head()

In [ ]:
# Create a categorical variable from a numeric condition
lake_features["depth_zone"] = "below_surface"
lake_features.loc[lake_features["depth"] == 0, "depth_zone"] = "surface"

lake_features[["lake_name", "sample_date", "depth", "depth_zone"]].head(10)

### Creating variables with `.assign()`

`.assign()` is useful in method chains because it returns a modified dataframe without changing the original object.

In [ ]:
# Create variables using assign()
lake_features_chained = (
    lake_core_renamed
    .assign(
        year=lambda df: df["sample_date"].dt.year,
        month=lambda df: df["sample_date"].dt.month,
        is_surface=lambda df: df["depth"] == 0
    )
)

lake_features_chained.head()

### Exercise 3

Starting from `lake_features`, create a new column named `season` using the month value.

Use this simple classification:

- December, January, February: `winter`
- March, April, May: `spring`
- June, July, August: `summer`
- September, October, November: `fall`

Hint: one approach is to create a dictionary that maps month numbers to season names, then use `.map()`.

In [ ]:
# Exercise 3

---
## 4. Sorting records

Sorting records helps us inspect data in a meaningful order.

The main pandas function is `.sort_values()`.

```python
df.sort_values("column_name")
```

Use `ascending=False` to sort from largest to smallest.

In [ ]:
# Sort by date
lake_features.sort_values("sample_date").head()

In [ ]:
# Sort by dissolved oxygen from highest to lowest
lake_features.sort_values("dissolved_oxygen", ascending=False).head()

In [ ]:
# Sort by multiple columns
lake_features.sort_values(["lake_name", "sample_date", "depth"]).head(10)

### Exercise 4

Create a dataframe named `deepest_records` that sorts the data from greatest depth to shallowest depth.

Display the columns `lake_name`, `sample_date`, `depth`, `temperature`, and `dissolved_oxygen` for the first 10 records.

In [ ]:
# Exercise 4

## 5. Grouped summaries

This section introduces the core concept. Focus on the examples below and experiment by modifying the code to reinforce your understanding.

In [ ]:
# Average temperature by lake
lake_features.groupby("lake_name")["temperature_C"].mean()

In [ ]:
# Count observations by lake
lake_features.groupby("lake_name")["temperature_C"].count()

### Named aggregations

For more organized output, use `.agg()` with named aggregations.

This creates a dataframe with clear column names.

In [ ]:
# Summarize several variables by lake
lake_summary = (
    lake_features
    .groupby("lake_name")
    .agg(
        n_records=("temperature_C", "count"),
        first_sample=("sample_date", "min"),
        last_sample=("sample_date", "max"),
        max_depth=("depth", "max"),
        mean_temperature=("temperature_C", "mean"),
        mean_dissolved_oxygen=("dissolved_oxygen", "mean")
    )
    .reset_index()
)

lake_summary.head()

### Grouping by more than one variable

You can group by multiple variables by passing a list of column names.

In [ ]:
# Create a surface-only dataset for seasonal summaries
surface_features = lake_features.loc[lake_features["depth"] == 0].copy()

# Add season using a month-to-season dictionary
season_map = {
    12: "winter", 1: "winter", 2: "winter",
    3: "spring", 4: "spring", 5: "spring",
    6: "summer", 7: "summer", 8: "summer",
    9: "fall", 10: "fall", 11: "fall"
}

surface_features["season"] = surface_features["month"].map(season_map)

surface_season_summary = (
    surface_features
    .groupby(["lake_name", "season"])
    .agg(
        n_records=("temperature_C", "count"),
        mean_temperature=("temperature_C", "mean"),
        mean_dissolved_oxygen=("dissolved_oxygen", "mean")
    )
    .reset_index()
)

surface_season_summary.head(12)

### Exercise 5

Create a grouped summary named `depth_zone_summary` that reports, for each `lake_name` and `depth_zone`:

- number of records
- mean temperature
- mean dissolved oxygen

Reset the index so the result is a regular dataframe.

In [ ]:
# Exercise 5

## 6. Joining datasets

This section introduces the core concept. Focus on the examples below and experiment by modifying the code to reinforce your understanding.

In [ ]:
# Create a small lake metadata table for demonstration
lake_metadata = pd.DataFrame({
    "lake_name": ["Allequash Lake", "Big Muskellunge Lake", "Crystal Lake", "Peter Lake", "Paul Lake", "Tuesday Lake", "Trout Lake"],
    "lake_group": ["reference", "reference", "reference", "experimental", "experimental", "experimental", "reference"],
    "landscape_position": ["upper", "lower", "upper", "upper", "upper", "upper", "lower"],
    "example_watershed_area_ha": [52.0, 98.0, 81.0, 14.0, 17.0, 11.0, 260.0]
})

lake_metadata

In [ ]:
# Left join: keep all rows in lake_summary and add matching metadata
lake_summary_with_metadata = pd.merge(
    lake_summary,
    lake_metadata,
    on="lake_name",
    how="left"
)

lake_summary_with_metadata.head()

### Checking joins

Always check a join. Common problems include:

- misspelled key values,
- unexpected duplicates,
- missing matches, and
- different capitalization or spacing.

One useful option is `indicator=True`, which adds a column showing whether each row matched both tables.

In [ ]:
# Diagnose matches between summary and metadata
join_check = pd.merge(
    lake_summary,
    lake_metadata,
    on="lake_name",
    how="left",
    indicator=True
)

join_check["_merge"].value_counts()

In [ ]:
# Show rows that did not match metadata
join_check.loc[join_check["_merge"] != "both", ["lake_name", "_merge"]]

### Exercise 6

Join `surface_season_summary` to `lake_metadata` using a left join.

Save the result as `surface_season_with_metadata`.

Then check how many records matched both datasets.

In [ ]:
# Exercise 6

---
## 7. Reshaping data between wide and long formats

Data can be stored in different shapes.

A **wide** dataset has multiple measurement variables stored in separate columns.

A **long** dataset stores measurement names in one column and measurement values in another column.

Long data is often easier for plotting, grouping, and modeling because each row represents one observation of one variable.

In [ ]:
# Start with a small wide dataset
surface_small_wide = surface_features.loc[
    :,
    ["lake_name", "sample_date", "temperature_C", "dissolved_oxygen"]
].head(10)

surface_small_wide

### Wide to long with `melt()`

Use `pd.melt()` or `.melt()` to convert columns into rows.

Important arguments:

- `id_vars`: columns that identify each observation and should remain as identifiers
- `value_vars`: columns that contain measured values to reshape
- `var_name`: name of the new column containing variable names
- `value_name`: name of the new column containing values

In [ ]:
# Convert from wide to long
surface_small_long = surface_small_wide.melt(
    id_vars=["lake_name", "sample_date"],
    value_vars=["temperature_C", "dissolved_oxygen"],
    var_name="measurement",
    value_name="value"
)

surface_small_long.head(15)

### Long to wide with `pivot_table()`

Use `.pivot_table()` to convert long data back to wide form.

Important arguments:

- `index`: columns that identify rows
- `columns`: the column whose values should become new column names
- `values`: the column whose values should fill the table
- `aggfunc`: how to handle duplicate combinations

In [ ]:
# Convert from long back to wide
surface_small_wide_again = (
    surface_small_long
    .pivot_table(
        index=["lake_name", "sample_date"],
        columns="measurement",
        values="value",
        aggfunc="mean"
    )
    .reset_index()
)

# Remove the columns axis name created by pivot_table
surface_small_wide_again.columns.name = None

surface_small_wide_again.head()

### Reshaping grouped summaries

Reshaping is especially useful after grouped summaries.

For example, suppose we want one row per lake and one column per season for mean surface temperature.

In [ ]:
# Create a wide table of mean surface temperature by lake and season
season_temperature_wide = (
    surface_season_summary
    .pivot_table(
        index="lake_name",
        columns="season",
        values="mean_temperature"
    )
    .reset_index()
)

season_temperature_wide.columns.name = None

season_temperature_wide.head()

In [ ]:
# Convert that seasonal summary back to long format
season_temperature_long = season_temperature_wide.melt(
    id_vars="lake_name",
    var_name="season",
    value_name="mean_temperature"
)

season_temperature_long.head(12)

### Exercise 7

Create a long version of `surface_season_summary` that keeps `lake_name`, `season`, and `n_records` as identifier columns, then reshapes these columns into a measurement/value pair:

- `mean_temperature`
- `mean_dissolved_oxygen`

Name the new columns `measurement` and `mean_value`.

In [ ]:
# Exercise 7

---
## 8. Pulling it together: an analysis-ready workflow

The following workflow demonstrates how several wrangling steps can be combined into one readable pipeline.

The goal is to create a lake-by-season summary of surface observations, joined to lake metadata, with clear variable names.

In [ ]:
analysis_ready_summary = (
    NTL_phys_data
    .rename(columns={
        "lakename": "lake_name",
        "sampledate": "sample_date",
        "dissolvedOxygen": "dissolved_oxygen"
    })
    .loc[lambda df: df["depth"] == 0]
    .assign(
        year=lambda df: df["sample_date"].dt.year,
        month=lambda df: df["sample_date"].dt.month,
        season=lambda df: df["month"].map(season_map)
    )
    .groupby(["lake_name", "season"])
    .agg(
        n_records=("temperature_C", "count"),
        mean_temperature=("temperature_C", "mean"),
        mean_dissolved_oxygen=("dissolved_oxygen", "mean")
    )
    .reset_index()
    .merge(lake_metadata, on="lake_name", how="left")
    .sort_values(["lake_name", "season"])
)

analysis_ready_summary.head(12)

### Exporting a processed dataset

When you create a useful intermediate dataset, save it to the `data/processed/` folder rather than overwriting the raw data.

This keeps the workflow reproducible:

- raw data stays unchanged,
- processed data can be recreated from code,
- notebook outputs are easier to review.

In [ ]:
# Export the analysis-ready summary
analysis_ready_summary.to_csv(
    processed_fldr / "NTL_surface_season_summary.csv",
    index=False
)

## Capstone exercise

Create a processed dataset that meets the following specifications:

1. Use only observations from Peter, Paul, and Tuesday Lakes.
2. Use only surface observations.
3. Keep only these variables from the raw dataset:
   - lake name
   - sample date
   - depth
   - temperature
   - dissolved oxygen
4. Rename the variables using `snake_case` names.
5. Create `year`, `month`, and `season` columns.
6. Remove records with missing values in temperature or dissolved oxygen.
7. Summarize by lake and season.
8. Report:
   - number of records
   - mean temperature
   - mean dissolved oxygen
9. Join the result to `lake_metadata`.
10. Sort the result by lake name and season.

Save the final dataframe as `peter_paul_tuesday_summary`.

In [ ]:
# Capstone exercise

## Key takeaways

This section introduces the core concept. Focus on the examples below and experiment by modifying the code to reinforce your understanding.